# 01 — RMSNorm (M1.1)

This notebook explains and verifies the RMSNorm layer used everywhere in Qwen3.
It imports the tested implementation from `qwen3_scratch.layers` instead of
redefining it. Runs on CPU in a few seconds; no checkpoints, no downloads.
The optional comparison with Hugging Face runs only if `transformers` is installed.

Full write-up: `docs/components/rmsnorm.md`.

## 1. Motivation and intuition

A deep residual network keeps adding sublayer outputs to the residual stream
$x \leftarrow x + f(x)$. The scale of $x$ drifts from layer to layer and token to
token, which makes each sublayer see inputs of unpredictable magnitude. A
normalisation layer in front of every sublayer (**pre-norm**) rescales each token
vector to a fixed "size" so attention and the MLP always receive well-conditioned
inputs. This stabilises training of very deep Transformers.

RMSNorm (Zhang & Sennrich, 2019) keeps only the **re-scaling** part of LayerNorm:
it divides by the root mean square and multiplies by a learned per-feature scale
$\gamma$. It does **not** subtract the mean and has **no bias**. That saves one
reduction per token and, empirically, loses no quality.

## 2. Mathematics

For one token vector $x \in \mathbb{R}^d$ (the last tensor dimension):

$$\operatorname{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon},\qquad
y_i=\frac{x_i}{\operatorname{RMS}(x)}\,\gamma_i$$

| step | operation | shape for input `(B, T, d)` |
|---|---|---|
| square | $x_i^2$ | `(B, T, d)` |
| mean over last dim | $m=\frac1d\sum_i x_i^2$ | `(B, T, 1)` (keepdim for broadcasting) |
| add eps, inverse sqrt | $1/\sqrt{m+\epsilon}$ | `(B, T, 1)` |
| normalise | $\hat x = x/\sqrt{m+\epsilon}$ | `(B, T, d)` |
| scale | $y=\gamma\odot\hat x$, $\gamma\in\mathbb{R}^d$ | `(B, T, d)` |

With $\gamma=1$ and $\epsilon\to0$, every output row has mean square exactly 1,
i.e. Euclidean norm $\sqrt d$.

## 3. Manual calculation with a 4-dimensional vector

In [ ]:
import math

import torch

from qwen3_scratch.layers import RMSNorm, rms_norm, rms_norm_textbook

x = [1.0, 2.0, 3.0, 4.0]
gamma = [0.5, 1.0, -1.0, 2.0]
eps = 1e-6

squares = [v * v for v in x]
mean_square = sum(squares) / len(x)
rms = math.sqrt(mean_square + eps)
x_hat = [v / rms for v in x]
y_manual = [g * v for g, v in zip(gamma, x_hat, strict=True)]

print("squares     :", squares)
print("mean square :", mean_square)
print("rms         :", rms)
print("x_hat       :", [round(v, 6) for v in x_hat])
print("y (manual)  :", [round(v, 6) for v in y_manual])
print("mean(x_hat^2):", sum(v * v for v in x_hat) / 4, "(= 7.5 / 7.500001)")

In [ ]:
norm = RMSNorm(4, eps=eps, dtype=torch.float64)
with torch.no_grad():
    norm.weight.copy_(torch.tensor(gamma, dtype=torch.float64))
y_module = norm(torch.tensor(x, dtype=torch.float64))
print("y (module)  :", y_module.tolist())
print(
    "max |module - manual| =",
    max(abs(a - b) for a, b in zip(y_module.tolist(), y_manual, strict=True)),
)

## 4. Implementation walkthrough

`RMSNorm` owns a single parameter `weight` of shape `(d,)` initialised to ones
(same name and init as Hugging Face `Qwen3RMSNorm`, no `1 + weight` offset).
`forward` calls the functional `rms_norm`, shown below. The comments mark the
three Qwen3-specific precision choices.

In [ ]:
import inspect

print(inspect.getsource(rms_norm))
print(RMSNorm(1024))

## 5. Shapes: hidden-state norm vs. per-head QK-Norm

In [ ]:
from qwen3_scratch.config import TINY_QWEN3 as cfg

B, T = 2, 5
hidden = torch.randn(B, T, cfg.hidden_size)
block_norm = RMSNorm(cfg.hidden_size, eps=cfg.rms_norm_eps)
print(
    "block norm :",
    tuple(hidden.shape),
    "->",
    tuple(block_norm(hidden).shape),
    "| weight",
    tuple(block_norm.weight.shape),
)

# QK-Norm: queries reshaped to (B, T, H, head_dim); normalise each head vector.
q = torch.randn(B, T, cfg.num_attention_heads, cfg.head_dim)
q_norm = RMSNorm(cfg.head_dim, eps=cfg.rms_norm_eps)
print(
    "q_norm     :",
    tuple(q.shape),
    "->",
    tuple(q_norm(q).shape),
    "| weight",
    tuple(q_norm.weight.shape),
    "(shared by all heads)",
)

per_head_ms = q_norm(q).pow(2).mean(-1)
print(
    "mean square per (b, t, head):",
    per_head_ms.shape,
    "min/max",
    round(per_head_ms.min().item(), 6),
    round(per_head_ms.max().item(), 6),
)

## 6. RMSNorm vs. LayerNorm

In [ ]:
torch.manual_seed(0)
v = torch.randn(64, dtype=torch.float64)
rms_layer = RMSNorm(64, dtype=torch.float64)
layer_norm = torch.nn.LayerNorm(64, eps=1e-6, elementwise_affine=False, dtype=torch.float64)

for name, f in [("RMSNorm", rms_layer), ("LayerNorm", layer_norm)]:
    shift = (f(v + 5.0) - f(v)).abs().max().item()
    scale = (f(3.0 * v) - f(v)).abs().max().item()
    print(f"{name:9s}  change after +5 shift: {shift:.2e}   after x3 scale: {scale:.2e}")

y_rms = rms_layer(v + 5.0)
print("RMSNorm output mean after shift:", round(y_rms.mean().item(), 4), "(mean is NOT removed)")

## 7. Comparison with the official Qwen3 behaviour

The Hugging Face `Qwen3RMSNorm.forward` does, in order: cast to float32 →
mean of squares → `x * rsqrt(var + eps)` → cast back to the input dtype →
multiply by `weight`. Our `rms_norm` follows the same order, so outputs should be
**bit-identical**. `torch.nn.RMSNorm` and the educational Appendix C code instead
multiply by the weight before casting back ("scale-then-cast"), which differs by
one bfloat16 rounding step on some elements.

In [ ]:
def scale_then_cast(x, w, eps):
    h = x.float()
    h = h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + eps)
    return (h * w.float()).to(x.dtype)


gen = torch.Generator().manual_seed(0)
d = 1024
xb = (2 * torch.randn(4, 128, d, generator=gen)).bfloat16()
wb = (1 + 0.2 * torch.randn(d, generator=gen)).bfloat16()
ours = RMSNorm(d, dtype=torch.bfloat16)
tnn = torch.nn.RMSNorm(d, eps=1e-6, dtype=torch.bfloat16)  # eps must be explicit!
with torch.no_grad():
    ours.weight.copy_(wb)
    tnn.weight.copy_(wb)
    y_ours, y_tnn, y_alt = ours(xb), tnn(xb), scale_then_cast(xb, wb, 1e-6)

print("torch.nn.RMSNorm == scale-then-cast :", torch.equal(y_tnn, y_alt))
print("elements where ours != torch.nn     :", f"{(y_ours != y_tnn).float().mean().item():.1%}")
print("max |ours - torch.nn|               :", (y_ours.float() - y_tnn.float()).abs().max().item())

try:
    from transformers.models.qwen3.modeling_qwen3 import Qwen3RMSNorm
except ImportError:
    print("transformers not installed -> HF comparison skipped (pip install -e '.[hf]')")
else:
    hf = Qwen3RMSNorm(d, eps=1e-6).to(torch.bfloat16)
    with torch.no_grad():
        hf.weight.copy_(wb)
        print("bit-identical to HF Qwen3RMSNorm    :", torch.equal(y_ours, hf(xb)))

## 8. Forward and backward

With $u=\partial L/\partial y$, $g=u\odot\gamma$ and $r=\sqrt{m+\epsilon}$:

$$\frac{\partial L}{\partial x}=\frac{g}{r}-\frac{x\,(g\cdot x)}{d\,r^{3}},\qquad
\frac{\partial L}{\partial \gamma}=\sum_{\text{positions}} u\odot\hat x$$

The second term removes the component of the gradient along $x$: RMSNorm is
(almost) scale invariant, so moving $x$ along itself barely changes the output.
Autograd builds this automatically from the primitive ops (`pow`, `mean`,
`rsqrt`, `mul`); we only check that it agrees with the formula.

In [ ]:
gen = torch.Generator().manual_seed(1)
xg = torch.randn(3, 16, generator=gen, dtype=torch.float64, requires_grad=True)
layer = RMSNorm(16, dtype=torch.float64)
with torch.no_grad():
    layer.weight.copy_(1 + 0.3 * torch.randn(16, generator=gen, dtype=torch.float64))
u = torch.randn(3, 16, generator=gen, dtype=torch.float64)
layer(xg).backward(u)

xd, gam = xg.detach(), layer.weight.detach()
r = torch.sqrt(xd.pow(2).mean(-1, keepdim=True) + 1e-6)
g = u * gam
dx = g / r - xd * (g * xd).sum(-1, keepdim=True) / (16 * r**3)
dgamma = (u * xd / r).sum(0)
print("max |autograd - formula| dx     :", (xg.grad - dx).abs().max().item())
print("max |autograd - formula| dgamma :", (layer.weight.grad - dgamma).abs().max().item())
print("x . dL/dx per row (eps-sized)   :", (xd * xg.grad).sum(-1).tolist())

ok = torch.autograd.gradcheck(
    lambda a, b: rms_norm(a, b, 1e-6), (xd.clone().requires_grad_(), gam.clone().requires_grad_())
)
print("gradcheck (float64)             :", ok)

## 9. Numerical error analysis

In [ ]:
gen = torch.Generator().manual_seed(2)
x64 = 2 * torch.randn(8, 1024, generator=gen, dtype=torch.float64)
w64 = 1 + 0.2 * torch.randn(1024, generator=gen, dtype=torch.float64)

print("dtype        max_abs     max_rel   (vs float64 textbook on the same rounded inputs)")
for dtype in (torch.float32, torch.bfloat16, torch.float16):
    xs, ws = x64.to(dtype), w64.to(dtype)
    y = rms_norm(xs, ws, 1e-6).double()
    ref = rms_norm_textbook(xs.double(), ws.double(), 1e-6)
    diff = (y - ref).abs()
    max_rel = (diff / ref.abs().clamp_min(1e-3)).max().item()
    name = str(dtype).split(".")[-1]
    print(f"{name:10s}{diff.max().item():12.3e}{max_rel:12.3e}")

print("\nbf16 unit roundoff 2^-8 =", 2**-8, "; one bf16 ulp relative <= 2^-7 =", 2**-7)

In [ ]:
# Why statistics are computed in float32: squares overflow float16 quickly.
big = torch.full((4,), 300.0, dtype=torch.float16)
print("float16 300^2           :", (big * big).tolist())
print(
    "naive fp16 RMSNorm      :",
    rms_norm_textbook(big, torch.ones(4, dtype=torch.float16), 1e-6).tolist(),
)
print("ours (fp32 statistics)  :", RMSNorm(4, dtype=torch.float16)(big).tolist())

# Why eps matters: it bounds the gain applied to near-zero vectors.
for scale in (1.0, 1e-2, 1e-3, 1e-4, 0.0):
    t = scale * torch.ones(8, dtype=torch.float64)
    out = RMSNorm(8, dtype=torch.float64)(t)
    print(f"input {scale:7.0e} -> output {out[0].item():.6f}")

## 10. Common mistakes and debugging tips

- **Wrong axis.** Normalise over the last (feature) dimension only; `mean()` without
  `dim=-1, keepdim=True` mixes tokens or breaks broadcasting.
- **Hidden dim not last.** A `(d, T)` layout silently normalises the wrong axis; our
  module raises if `x.shape[-1] != hidden_size`.
- **Subtracting the mean / adding a bias.** That is LayerNorm, not RMSNorm.
- **`sqrt(mean) + eps` instead of `sqrt(mean + eps)`.** Different numbers near zero.
- **Forgetting eps defaults differ.** `torch.nn.RMSNorm(eps=None)` uses
  `torch.finfo(dtype).eps`, not Qwen3's `1e-6`.
- **Weight offset.** Some models (e.g. Gemma) store `w` and use `1 + w`, initialised
  to zero. Qwen3 stores `w` directly, initialised to one.
- **Low-precision statistics.** Computing the mean of squares in fp16/bf16 loses
  precision or overflows; upcast to fp32.
- **Cast order.** For bit-exact parity with HF, cast back to the input dtype
  *before* multiplying by the weight.
- **QK-Norm shape.** `q_norm`/`k_norm` have shape `(head_dim,)` and act per head,
  after reshaping to `(..., heads, head_dim)` and before RoPE.

Debugging: compare against `rms_norm_textbook` in float64 first, then against HF;
check `y.pow(2).mean(-1)` is about `1` with unit weights; inspect dtypes at each step.